# Detecting Structural Distress at Scale — v4 (Clay Embeddings + RandomForest)

**Course:** MUSA 6500 — Geospatial Machine Learning in Remote Sensing
**Authors:** Jason Fan, Henry Sywulak-Herr

## Why this approach

Clay's official downstream-classification recipe
([docs/finetune/finetune-on-embeddings.ipynb](https://clay-foundation.github.io/model/finetune/finetune-on-embeddings.html))
is **NOT neural-network fine-tuning**. It's:

1. Run Clay as a pure feature extractor, cache embeddings per chip
2. Train `sklearn.ensemble.RandomForestClassifier` on those cached embeddings

The tutorial achieves 90% accuracy on 216 marina-detection samples. No training loops,
no AMP, no gradient-based fine-tuning at all.

Previous attempts (main_v2.ipynb) used LLRD and neural-network heads, which is
outside Clay's documented use pattern for classification. This notebook follows the
official pattern exactly.

## Key Clay-specific settings

- **IMAGE_SIZE = 256** (Clay's native resolution → 32×32 = 1024 patch tokens)
- **ClayMAEModule.load_from_checkpoint** with explicit `model_size`, `dolls`, `doll_weights`
  as prescribed by the embeddings tutorial
- **LINZ-aligned conditioning** (closest pretraining platform to 0.3 m aerial RGB)
- **Zero time/latlon** (sanctioned by Clay docs: *"can be set to zero if not available"*)
- **CLS token readout** (matches claymodel/finetune/classify/factory.py)

## 1. Environment setup

In [2]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import math
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
  GPU: NVIDIA A100-SXM4-80GB


In [3]:
import sys, os, subprocess

IS_COLAB = os.path.exists('/content')

if IS_COLAB:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/Clay-foundation/model.git',
        'rioxarray', 'geopandas', 'tqdm', 'scikit-learn',
    ], check=True)

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_FOLDER = '/content/drive/MyDrive/folder'
    if DRIVE_FOLDER not in sys.path:
        sys.path.insert(0, DRIVE_FOLDER)

    DRIVE_BASE = Path(DRIVE_FOLDER)
    PROJECT_ROOT = Path('/content')
    print(f'Drive mounted. Project folder: {DRIVE_FOLDER}')
else:
    PROJECT_ROOT = Path('.').resolve()
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    DRIVE_BASE = PROJECT_ROOT
    print(f'Running locally. Project root: {PROJECT_ROOT}')

CHIP_DIR = Path('/content/chips_local') if IS_COLAB else PROJECT_ROOT / 'data' / 'chips'
CHIP_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_BASE / 'data').mkdir(parents=True, exist_ok=True)
print(f'Chip cache: {CHIP_DIR}')

Mounted at /content/drive
Drive mounted. Project folder: /content/drive/MyDrive/folder
Chip cache: /content/chips_local


## 2. Data loading

In [4]:
from load_imagery import open_imagery

src = open_imagery()

[load_imagery] Opened: https://musa-650.s3.amazonaws.com/phl_aerial_0.3m.tif
  CRS   : EPSG:2272
  Shape : (3, 123383, 109972)  (bands × rows × cols)
  Res   : (0.984252, -0.984252) (x_res, y_res)


In [ ]:
from load_building_footprints import load_building_footprints, fetch_geojson

VECTOR_DIR = Path('data/vector')
CRS = 'EPSG:2272'

parcels = load_building_footprints()
print(f'Parcels: {len(parcels):,}')

PERMIT_URL = ('https://hub.arcgis.com/api/v3/datasets/'
              '8d18914ff740444793937d8724c64da8_0/downloads/data'
              '?format=geojson&spatialRefId=4326&where=1%3D1')
permits = fetch_geojson('permits', PERMIT_URL, VECTOR_DIR).to_crs(CRS)
print(f'Permits: {len(permits):,}')

[load_building_footprints] Fetching parcels from https://opendata.arcgis.com/datasets/84baed491de44f539889f2af178ad85c_0.geojson ...
  → cached to data/vector/parcels.geojson


In [ ]:
from load_labels import load_labels, plot_labels

parcels_labeled = load_labels(parcels, permits=permits)

assert 'label' in parcels_labeled.columns
assert 'label_permit_flagged' in parcels_labeled.columns
print("\nLabel columns OK:",
      [c for c in parcels_labeled.columns if 'label' in c.lower()])

plot_labels(parcels_labeled)

## 3. Train / test split strategy

In [ ]:
NATURAL_TEST_FRAC = 0.10

rng = np.random.default_rng(SEED)
n = len(parcels_labeled)
natural_idx = rng.choice(n, size=int(n * NATURAL_TEST_FRAC), replace=False)
natural_mask = np.zeros(n, dtype=bool)
natural_mask[natural_idx] = True

natural_test = parcels_labeled[natural_mask].copy().reset_index(drop=True)
remaining    = parcels_labeled[~natural_mask].copy()

natural_test['binary_label'] = (natural_test['label'] > 0).astype(int)

print(f"Natural-distribution holdout: {len(natural_test):,}")
print(f"  Stable     : {(natural_test['binary_label']==0).sum():,}")
print(f"  Distressed : {(natural_test['binary_label']==1).sum():,}")
print(f"  Real prior : {natural_test['binary_label'].mean():.4f}")

In [ ]:
distressed = remaining[remaining['label'].isin([1, 2])].copy()
num_stable = len(distressed) * 2
stable = (remaining[remaining['label'] == 0]
          .sample(n=num_stable, random_state=SEED).copy())

balanced_parcels = (pd.concat([distressed, stable])
                    .sample(frac=1, random_state=SEED)
                    .reset_index(drop=True))
balanced_parcels['label'] = balanced_parcels['label'].replace({2: 1})

print(f"Balanced training pool: {len(balanced_parcels):,}")
print(balanced_parcels['label'].value_counts())

(DRIVE_BASE / 'data' / 'balanced_parcels.geojson').unlink(missing_ok=True)
balanced_parcels[['geometry', 'label']].to_file(
    DRIVE_BASE / 'data' / 'balanced_parcels.geojson', driver='GeoJSON'
)

## 4. Clay checkpoint + metadata

In [ ]:
import urllib.request

CKPT_PATH = DRIVE_BASE / "data" / "clay-v1.5.ckpt"
METADATA_PATH = DRIVE_BASE / "data" / "clay_metadata.yaml"

if not CKPT_PATH.exists():
    CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
    print("Downloading Clay v1.5 checkpoint (~3 GB)…")
    urllib.request.urlretrieve(
        "https://huggingface.co/made-with-clay/Clay/resolve/main/v1.5/clay-v1.5.ckpt",
        CKPT_PATH,
    )
else:
    print(f"Clay checkpoint: {CKPT_PATH} ({CKPT_PATH.stat().st_size/1e9:.1f} GB)")

if not METADATA_PATH.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Clay-foundation/model/main/configs/metadata.yaml",
        METADATA_PATH,
    )
print(f"Clay metadata: {METADATA_PATH}")

## 5. Chip cache — 130 ft (~40 m) fixed windows

ConvNeXt's 224×224 pad+resize logic maps to Clay's 256×256 in the dataset.
Existing 130-ft chips from previous sessions work — no rechip needed.

In [ ]:
from tqdm import tqdm

CHIP_FEET = 130.0


def read_fixed_window(src, geom, size_feet: float = CHIP_FEET):
    cx, cy = geom.centroid.x, geom.centroid.y
    half = size_feet / 2
    chip = src.sel(
        x=slice(cx - half, cx + half),
        y=slice(cy + half, cy - half),
    ).values
    if chip.ndim == 3 and chip.shape[0] >= 3:
        chip = chip[:3]
    else:
        raise ValueError(f"Unexpected chip shape: {chip.shape}")
    return chip.astype(np.float32)


def precache_chips(gdf, src, chip_dir, desc="Caching chips"):
    chip_dir.mkdir(parents=True, exist_ok=True)
    n_fallback = 0
    for idx, row in tqdm(gdf.iterrows(), total=len(gdf), desc=desc):
        out = chip_dir / f"{idx}.npy"
        if out.exists():
            continue
        try:
            chip = read_fixed_window(src, row.geometry)
            if chip.shape[1] < 10 or chip.shape[2] < 10:
                raise ValueError(f"Chip too small: {chip.shape}")
            np.save(out, chip)
        except Exception:
            np.save(out, np.zeros((3, 64, 64), dtype=np.float32))
            n_fallback += 1
    print(f"{desc} done. {n_fallback}/{len(gdf)} fallback chips.")

In [ ]:
REBUILD_CHIPS = False

DRIVE_TAR = DRIVE_BASE / "data" / "chips.tar"
LOCAL_TAR = Path("/content/chips.tar") if IS_COLAB else PROJECT_ROOT / "data" / "chips.tar"

if REBUILD_CHIPS and CHIP_DIR.exists():
    print(f"REBUILD_CHIPS=True — wiping {CHIP_DIR}")
    shutil.rmtree(CHIP_DIR)
    CHIP_DIR.mkdir(parents=True, exist_ok=True)

existing = len(list(CHIP_DIR.glob('*.npy')))
needed = len(balanced_parcels)

if existing >= needed * 0.98:
    print(f"Chips already cached: {existing:,} files in {CHIP_DIR}")
elif DRIVE_TAR.exists() and not REBUILD_CHIPS:
    print(f"Restoring chips from {DRIVE_TAR}…")
    if IS_COLAB:
        os.system(f'cp "{DRIVE_TAR}" "{LOCAL_TAR}"')
        os.system(f'tar -xf "{LOCAL_TAR}" -C /content/')
    else:
        os.system(f'tar -xf "{DRIVE_TAR}" -C "{CHIP_DIR.parent}"')
    existing = len(list(CHIP_DIR.glob('*.npy')))
    print(f"Restored: {existing:,} chips")
    if existing < needed * 0.95:
        print(f"Tarball incomplete — precaching missing chips…")
        precache_chips(balanced_parcels, src, CHIP_DIR)
else:
    print(f"Precaching {needed:,} chips from S3 (~20-30 min)…")
    precache_chips(balanced_parcels, src, CHIP_DIR)

if not DRIVE_TAR.exists() and IS_COLAB:
    print(f"Creating tarball for future sessions…")
    os.system(f'tar -cf "{LOCAL_TAR}" -C /content chips_local')
    os.system(f'cp "{LOCAL_TAR}" "{DRIVE_TAR}"')

## 6. Dataset — 256×256 output for Clay's native resolution

In [ ]:
IMAGE_SIZE = 256  # Clay's native pretraining resolution

# LINZ band stats from Clay's metadata.yaml — 0.5m aerial RGB
CLAY_RGB_MEAN = torch.tensor([89.96, 99.46, 89.51]).view(3, 1, 1) / 255.0
CLAY_RGB_STD  = torch.tensor([41.83, 36.96, 31.45]).view(3, 1, 1) / 255.0


def _pad_to_square_and_resize(arr, size=IMAGE_SIZE):
    t = torch.from_numpy(arr)
    _, H, W = t.shape
    side = max(H, W)
    pad_h, pad_w = side - H, side - W
    t = F.pad(t, (pad_w // 2, pad_w - pad_w // 2,
                  pad_h // 2, pad_h - pad_h // 2), value=0.0)
    t = F.interpolate(t.unsqueeze(0), size=(size, size),
                      mode='bilinear', align_corners=False).squeeze(0)
    return t


class ParcelDataset(Dataset):
    """Minimal dataset for embedding extraction — no augmentations needed."""

    def __init__(self, gdf, chip_dir=None, label_col='label'):
        self.gdf = gdf
        self.chip_dir = Path(chip_dir) if chip_dir is not None else CHIP_DIR
        self.label_col = label_col

    def __len__(self):
        return len(self.gdf)

    def __getitem__(self, idx):
        row = self.gdf.iloc[idx]
        label = int(row[self.label_col])
        chip_id = row.name

        chip_path = self.chip_dir / f"{chip_id}.npy"
        try:
            arr = np.load(chip_path).astype(np.float32)
            arr = np.clip(arr / 255.0, 0.0, 1.0)
        except Exception:
            arr = np.zeros((3, IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)

        img = _pad_to_square_and_resize(arr, size=IMAGE_SIZE)
        img = (img - CLAY_RGB_MEAN) / CLAY_RGB_STD
        return img, torch.tensor(label, dtype=torch.long)

## 7. Load Clay — official ClayMAEModule with prescribed args

Arguments match the [embeddings tutorial](https://clay-foundation.github.io/model/tutorials/embeddings.html) exactly.

In [ ]:
from claymodel.module import ClayMAEModule

print("Loading Clay v1.5 large …")
clay_module = ClayMAEModule.load_from_checkpoint(
    checkpoint_path=str(CKPT_PATH),
    model_size="large",                                  # ← official arg from tutorial
    metadata_path=str(METADATA_PATH),
    dolls=[16, 32, 64, 128, 256, 768, 1024],             # ← MRL levels from tutorial
    doll_weights=[1, 1, 1, 1, 1, 1, 1],
    mask_ratio=0.0,
    shuffle=False,
    strict=False,
)
clay_module.eval()
clay_module = clay_module.to(DEVICE)

clay_encoder = clay_module.model.encoder
print(f"  encoder: {type(clay_encoder).__name__}")
print(f"  params: {sum(p.numel() for p in clay_encoder.parameters()):,}")

# LINZ-aligned conditioning (zero time/latlon per Clay docs)
RGB_WAVES = torch.tensor([0.635, 0.555, 0.465])
RGB_GSD   = torch.tensor(0.5)

## 8. Extract embeddings

Run Clay once per chip and cache the 1024-dim CLS token as a numpy array.
This is the "heavy" step — ~5–10 min on A100 for ~21K chips.

In [ ]:
@torch.no_grad()
def extract_embeddings(loader, encoder, device=DEVICE):
    all_embeds = []
    all_labels = []
    for images, labels in tqdm(loader, desc="Extracting"):
        images = images.to(device, non_blocking=True)
        B = images.shape[0]
        datacube = {
            "pixels": images,
            "time":   torch.zeros(B, 4, device=device),
            "latlon": torch.zeros(B, 4, device=device),
            "gsd":    RGB_GSD.to(device),
            "waves":  RGB_WAVES.to(device),
        }
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=(device == 'cuda')):
            embeddings, *_ = encoder(datacube)
            cls = embeddings[:, 0, :].float()          # CLS token, [B, 1024]
        all_embeds.append(cls.cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_embeds), np.concatenate(all_labels)

In [ ]:
from sklearn.model_selection import train_test_split

# Use the full balanced pool — no augmentation, just pure feature extraction
full_loader = DataLoader(
    ParcelDataset(balanced_parcels),
    batch_size=64, shuffle=False, num_workers=0,
    pin_memory=(DEVICE == 'cuda'),
)

EMBED_PATH = DRIVE_BASE / "data" / "clay_embeddings_balanced.npz"

if EMBED_PATH.exists():
    print(f"Loading cached embeddings: {EMBED_PATH}")
    data = np.load(EMBED_PATH)
    X_all, y_all = data['X'], data['y']
else:
    print(f"Extracting Clay embeddings for {len(balanced_parcels):,} chips …")
    X_all, y_all = extract_embeddings(full_loader, clay_encoder)
    np.savez(EMBED_PATH, X=X_all, y=y_all)
    print(f"Saved: {EMBED_PATH}")

print(f"\nEmbedding shape: {X_all.shape}")
print(f"Label shape:     {y_all.shape}")
print(f"Label balance:   {np.bincount(y_all)}")

# Feature sanity check — are embeddings discriminative?
std_across = X_all.std(axis=0).mean()
std_within = X_all.std(axis=1).mean()
print(f"\nEmbedding quality:")
print(f"  std_within_sample = {std_within:.3f}")
print(f"  std_across_batch  = {std_across:.4f}  (need > 0.05 for classifier to work)")

In [ ]:
# 80/20 split on the balanced embeddings
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=SEED,
)
print(f"Train: {X_train.shape}  Val: {X_val.shape}")

## 9. Train RandomForest classifier

Matches the official [finetune-on-embeddings tutorial](https://clay-foundation.github.io/model/finetune/finetune-on-embeddings.html) exactly.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve,
)

# Tutorial uses defaults; we bump n_estimators for a slightly stronger model
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1,
)

print("Training RandomForest on Clay embeddings …")
rf.fit(X_train, y_train)
print("Done.")

# Default-threshold predictions (RF uses 0.5)
y_pred  = rf.predict(X_val)
y_proba = rf.predict_proba(X_val)[:, 1]   # probability of class 1

print(f"\n--- Validation report (RF default threshold 0.5) ---")
print(classification_report(y_val, y_pred,
                            target_names=["Stable", "Distressed"], zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_val, y_pred, display_labels=["Stable", "Distressed"], cmap="Blues",
)
plt.title("Validation Confusion Matrix")
plt.tight_layout()
plt.show()

## 10. Threshold tuning

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)
f1s = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = f1s[:-1].argmax()
THRESHOLD_TUNED = float(thresholds[best_idx])

print(f"Best-F1 threshold: {THRESHOLD_TUNED:.3f}")
print(f"  P={precisions[best_idx]:.3f}  R={recalls[best_idx]:.3f}  F1={f1s[best_idx]:.3f}\n")

print(f"{'thr':>5}  {'recall':>7}  {'prec':>7}  {'f1':>6}  {'pos_rate':>8}")
for thr in [0.20, 0.30, 0.33, 0.40, 0.50, 0.60, 0.70]:
    p = (y_proba > thr).astype(int)
    tp = int(((p == 1) & (y_val == 1)).sum())
    fp = int(((p == 1) & (y_val == 0)).sum())
    fn = int(((p == 0) & (y_val == 1)).sum())
    rec  = tp / max(tp + fn, 1)
    prec = tp / max(tp + fp, 1)
    f1   = 2 * prec * rec / max(prec + rec, 1e-9)
    print(f"{thr:5.2f}  {rec:7.3f}  {prec:7.3f}  {f1:6.3f}  {p.mean():8.2f}")

print(f"\n--- At tuned threshold {THRESHOLD_TUNED:.3f} ---")
print(classification_report(y_val, (y_proba > THRESHOLD_TUNED).astype(int),
                            target_names=["Stable", "Distressed"], zero_division=0))

## 11. Natural-distribution holdout evaluation

In [ ]:
natural_chip_dir = Path("/content/natural_chips") if IS_COLAB else PROJECT_ROOT / "data" / "natural_chips"
natural_chip_dir.mkdir(parents=True, exist_ok=True)

print(f"Chipping {len(natural_test):,} natural-distribution parcels…")
precache_chips(natural_test, src, natural_chip_dir, desc="Natural chips")

NAT_EMBED_PATH = DRIVE_BASE / "data" / "clay_embeddings_natural.npz"

if NAT_EMBED_PATH.exists():
    data = np.load(NAT_EMBED_PATH)
    X_nat, y_nat = data['X'], data['y']
    print(f"Loaded cached: {NAT_EMBED_PATH}")
else:
    nat_loader = DataLoader(
        ParcelDataset(natural_test, chip_dir=natural_chip_dir, label_col='binary_label'),
        batch_size=64, shuffle=False, num_workers=0,
        pin_memory=(DEVICE == 'cuda'),
    )
    print(f"Extracting Clay embeddings for {len(natural_test):,} natural-dist chips …")
    X_nat, y_nat = extract_embeddings(nat_loader, clay_encoder)
    np.savez(NAT_EMBED_PATH, X=X_nat, y=y_nat)

y_nat_proba = rf.predict_proba(X_nat)[:, 1]
y_nat_pred  = (y_nat_proba > THRESHOLD_TUNED).astype(int)

print(f"\nNatural-distribution test (threshold={THRESHOLD_TUNED:.3f}):")
print(f"  n={len(y_nat):,}  true prior={np.mean(y_nat):.4f}")
print(f"  predicted positive rate: {np.mean(y_nat_pred):.4f}")
print(classification_report(y_nat, y_nat_pred,
                            target_names=["Stable", "Distressed"], zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_nat, y_nat_pred,
    display_labels=["Stable", "Distressed"], cmap="Oranges",
)
plt.title("Natural-Distribution Confusion Matrix")
plt.tight_layout()
plt.show()

## 11b. Visualize predictions like Clay tutorials

Four plot types from Clay's official tutorials, adapted to this classifier:

- **Chip grids** — TP / FN / FP / TN panels with colored borders (pattern from `reconstruction.ipynb`)
- **t-SNE of CLS embeddings** — 2-D projection colored by class (adapts the PCA scatter from `wall-to-wall.ipynb`)
- **Nearest-neighbor similarity search** — cosine-NN in 1024-dim space, grid display (pattern from `inference.ipynb`)
- **Geographic prediction map** — chips colored by distress probability (matplotlib version of the lonboard map from `finetune-on-embeddings.ipynb`)

In [ ]:
# --- Chip display helpers ---------------------------------------------------
def load_display_chip(parcel_idx, chip_dir=CHIP_DIR):
    """Load a cached .npy chip and return a displayable (H, W, 3) array in [0,1]."""
    path = Path(chip_dir) / f"{parcel_idx}.npy"
    arr = np.load(path).astype(np.float32) / 255.0
    return np.clip(arr, 0.0, 1.0).transpose(1, 2, 0)


def denormalize_clay(img_norm):
    """Invert the CLAY_RGB_MEAN/STD z-score used by ParcelDataset."""
    t = img_norm * CLAY_RGB_STD + CLAY_RGB_MEAN
    return torch.clamp(t, 0.0, 1.0).permute(1, 2, 0).cpu().numpy()


# Reproduce the exact val split on positional indices so we know which chip
# files (named by balanced_parcels index) belong to the val set.
idx_all = np.arange(len(balanced_parcels))
_, idx_val_pos, _, _ = train_test_split(
    idx_all, y_all, test_size=0.2, stratify=y_all, random_state=SEED,
)
print(f"Val positions recovered: {idx_val_pos.shape}  "
      f"(matches y_val length: {len(y_val)})")

In [ ]:
def plot_chip_grid(parcel_ids, probs, title, color, n=16, chip_dir=CHIP_DIR):
    parcel_ids = np.asarray(parcel_ids)
    probs = np.asarray(probs)
    if len(parcel_ids) == 0:
        print(f"[{title}] no samples"); return
    order = np.argsort(-probs)
    show_ids, show_p = parcel_ids[order][:n], probs[order][:n]

    ncol = 8
    nrow = max(1, math.ceil(len(show_ids) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(ncol * 2, nrow * 2.2))
    axs = np.atleast_2d(axs).flatten()
    for ax, pid, p in zip(axs, show_ids, show_p):
        try:
            ax.imshow(load_display_chip(pid, chip_dir))
        except FileNotFoundError:
            ax.text(0.5, 0.5, "missing", ha='center', va='center')
        ax.set_title(f"#{pid}  p={p:.2f}", fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_edgecolor(color); s.set_linewidth(3)
    for ax in axs[len(show_ids):]:
        ax.set_axis_off()
    fig.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()


y_pred_tuned = (y_proba > THRESHOLD_TUNED).astype(int)
tp = (y_val == 1) & (y_pred_tuned == 1)
fn = (y_val == 1) & (y_pred_tuned == 0)
fp = (y_val == 0) & (y_pred_tuned == 1)
tn = (y_val == 0) & (y_pred_tuned == 0)

print(f"Val outcomes @ threshold {THRESHOLD_TUNED:.2f}:  "
      f"TP={tp.sum()}  FN={fn.sum()}  FP={fp.sum()}  TN={tn.sum()}")

plot_chip_grid(idx_val_pos[tp], y_proba[tp],
               "True Positives — highest-confidence distressed hits",
               color="#2ca02c")
plot_chip_grid(idx_val_pos[fn], 1 - y_proba[fn],
               "False Negatives — distressed parcels we missed",
               color="#d62728")
plot_chip_grid(idx_val_pos[fp], y_proba[fp],
               "False Positives — stable parcels flagged as distressed",
               color="#ff7f0e")
plot_chip_grid(idx_val_pos[tn], 1 - y_proba[tn],
               "True Negatives — highest-confidence stable calls",
               color="#4a7cbf")

In [ ]:
from sklearn.manifold import TSNE

print("Running t-SNE on 1024-dim Clay CLS embeddings (~30-60 s)…")
tsne = TSNE(n_components=2, perplexity=30, init="pca",
            learning_rate="auto", random_state=SEED)
emb2d = tsne.fit_transform(X_val)

fig, ax = plt.subplots(figsize=(9, 7))
for cls, name, c, marker in [
    (0, "Stable",     "#4a7cbf", "o"),
    (1, "Distressed", "#d62728", "^"),
]:
    m = y_val == cls
    ax.scatter(emb2d[m, 0], emb2d[m, 1], s=14, alpha=0.55,
               c=c, marker=marker, edgecolor="none", label=f"{name} (n={m.sum()})")
ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
ax.set_title("Clay v1.5 CLS embeddings — t-SNE projection (validation)")
ax.legend(loc="best")
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=7, metric="cosine")
nn.fit(X_all)

rng_viz = np.random.default_rng(SEED)
distressed_positions = np.where(y_all == 1)[0]
queries = rng_viz.choice(distressed_positions, size=3, replace=False)

for q in queries:
    dists, idxs = nn.kneighbors(X_all[q:q + 1])
    neighbors, ds = idxs[0], dists[0]

    fig, axs = plt.subplots(1, 7, figsize=(18, 3.2))
    for k, (ax, nb, d) in enumerate(zip(axs, neighbors, ds)):
        ax.imshow(load_display_chip(nb))
        ax.set_xticks([]); ax.set_yticks([])
        lbl = "distressed" if y_all[nb] == 1 else "stable"
        ax.set_title(("QUERY\n" if k == 0 else f"cos-dist={d:.2f}\n") + lbl,
                     fontsize=9)
        border = "#d62728" if y_all[nb] == 1 else "#4a7cbf"
        for s in ax.spines.values():
            s.set_edgecolor(border); s.set_linewidth(3)
    fig.suptitle(f"Nearest neighbors in Clay embedding space — query chip #{q}",
                 y=1.05)
    plt.tight_layout(); plt.show()

In [ ]:
# Attach predictions to the natural-distribution holdout and map them.
nat_plot = natural_test.copy()
nat_plot["distress_score"] = y_nat_proba
nat_plot["pred"]           = y_nat_pred
nat_plot["outcome"] = np.select(
    [
        (nat_plot["binary_label"] == 1) & (nat_plot["pred"] == 1),
        (nat_plot["binary_label"] == 1) & (nat_plot["pred"] == 0),
        (nat_plot["binary_label"] == 0) & (nat_plot["pred"] == 1),
    ],
    ["TP", "FN", "FP"],
    default="TN",
)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

nat_plot.plot(
    column="distress_score", cmap="Reds",
    markersize=3, ax=axes[0], legend=True,
    legend_kwds={"label": "Predicted distress probability", "shrink": 0.55},
)
axes[0].set_title("Predicted distress probability\n(natural-distribution holdout)")
axes[0].set_axis_off()

palette = {"TP": "#2ca02c", "FN": "#d62728", "FP": "#ff7f0e", "TN": "#d0d0d0"}
for cls, c in palette.items():
    sub = nat_plot[nat_plot["outcome"] == cls]
    if len(sub):
        sub.plot(ax=axes[1], color=c, markersize=3,
                 label=f"{cls} (n={len(sub):,})")
axes[1].set_title(f"Classifier outcomes @ threshold {THRESHOLD_TUNED:.2f}")
axes[1].legend(loc="lower right", fontsize=9)
axes[1].set_axis_off()

plt.tight_layout(); plt.show()

## 11c. RGB change features — 2024 vs 2025 source.coop imagery

Source: [nlebovits/phl-aerial-imagery](https://source.coop/nlebovits/phl-aerial-imagery).
954 COG tiles per year at 3-inch GSD in EPSG:2272 with a STAC-GeoParquet
index. Five features per parcel:

- `mean_abs_diff` — overall pixel change
- `p95_abs_diff` — tail of the pixel-change distribution (catches localized damage)
- `ssim` — structural similarity (low = big layout change)
- `dark_new_frac` — fraction of pixels that went from bright → dark (exposed roof cavity)
- `pixel_count` — chip size, used only to detect coverage failures

Extraction is ~10–20 min for balanced_parcels and ~30–60 min for the natural
holdout over a typical home connection. Features are cached to .npz.

In [ ]:
import os
os.environ["GDAL_HTTP_USERAGENT"] = "Mozilla/5.0 (compatible; roof-collapse/1.0)"
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif,.tiff,.TIF,.TIFF"
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"  # speeds up /vsicurl/

In [ ]:
# --- Inlined rgb_change module with URL-pattern auto-detection ---------------
import os
import urllib.parse
import urllib.request
from pathlib import Path as _P

import rasterio
from shapely.geometry import box as _box

# Cloudflare blocks default Python/GDAL user agents. Set a browser-like UA
# before any HTTP call (parquet fetch AND /vsicurl/ COG reads).
_UA = "Mozilla/5.0 (compatible; roof-collapse/1.0)"
os.environ["GDAL_HTTP_USERAGENT"] = _UA
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif,.tiff,.TIF,.TIFF"
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"

SOURCE_COOP_BASE = "https://data.source.coop/nlebovits/phl-aerial-imagery"
PHILLY_CRS = "EPSG:2272"

CHANGE_CACHE_DIR = DRIVE_BASE / "data" / "source_coop"
CHANGE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

!pip install -q pyarrow


# ---------------------------------------------------------------------------
# URL pattern auto-detection
# ---------------------------------------------------------------------------
_URL_PATTERNS = [
    lambda base, year, iid, href: f"{base}/{year}/{iid}/{href}",
    lambda base, year, iid, href: f"{base}/{year}/{iid.upper()}/{href}",
    lambda base, year, iid, href: f"{base}/{year}/{href}",
    lambda base, year, iid, href: f"{base}/{year}/data/{iid}/{href}",
    lambda base, year, iid, href: f"{base}/{year}/data/{href}",
]


def _head_ok(url):
    req = urllib.request.Request(url, method="HEAD", headers={"User-Agent": _UA})
    try:
        with urllib.request.urlopen(req, timeout=10) as r:
            return r.status == 200
    except Exception:
        return False


def _find_url_pattern(year, sample_row):
    links = sample_row.get("links")
    if links is None:
        links = []
    for link in links:
        if isinstance(link, dict) and link.get("rel") == "self":
            self_href = link.get("href")
            if self_href and self_href.startswith("http"):
                asset_href = sample_row["assets"]["data"]["href"]
                resolved = urllib.parse.urljoin(self_href, asset_href)
                if _head_ok(resolved):
                    print(f"Using 'self' link resolution. Example: {resolved}")
                    sample_id = sample_row["id"]
                    return lambda base, y, iid, href: urllib.parse.urljoin(
                        self_href.replace(sample_id, iid), href
                    )

    href = sample_row["assets"]["data"]["href"].lstrip("./")
    iid  = sample_row["id"]
    for pattern in _URL_PATTERNS:
        candidate = pattern(SOURCE_COOP_BASE, year, iid, href)
        if _head_ok(candidate):
            print(f"Working URL pattern. Example: {candidate}")
            return pattern

    raise RuntimeError(
        f"No URL pattern matched for year {year}. "
        f"Sample id={iid!r}, href={href!r}. "
        f"Inspect df.iloc[0]['links'] manually."
    )


# ---------------------------------------------------------------------------
# Tile index
# ---------------------------------------------------------------------------
def load_tile_index(year, cache_dir=CHANGE_CACHE_DIR):
    cache_dir = _P(cache_dir); cache_dir.mkdir(parents=True, exist_ok=True)
    parquet_path = cache_dir / f"items_{year}.parquet"
    if not parquet_path.exists():
        url = f"{SOURCE_COOP_BASE}/{year}/items.parquet"
        print(f"Fetching tile index: {url}")
        req = urllib.request.Request(url, headers={"User-Agent": _UA})
        with urllib.request.urlopen(req) as resp, open(parquet_path, "wb") as f:
            f.write(resp.read())

    import pyarrow.parquet as pq
    df = pq.read_table(parquet_path).to_pandas()

    pattern_fn = _find_url_pattern(year, df.iloc[0])

    records = []
    for _, row in df.iterrows():
        href = row["assets"]["data"]["href"].lstrip("./")
        records.append({
            "url": pattern_fn(SOURCE_COOP_BASE, year, row["id"], href),
            "geometry": _box(*row["proj:bbox"]),
        })
    return gpd.GeoDataFrame(records, crs=PHILLY_CRS)


# ---------------------------------------------------------------------------
# Chip extraction
# ---------------------------------------------------------------------------
def _window_read(tile_url, bounds):
    with rasterio.open(f"/vsicurl/{tile_url}") as src:
        win = src.window(*bounds)
        arr = src.read(indexes=[1, 2, 3], window=win, boundless=True, fill_value=0)
    return np.transpose(arr, (1, 2, 0))


def _chip_pair(geom, idx_a, idx_b, buffer_ft=5.0):
    minx, miny, maxx, maxy = geom.buffer(buffer_ft).bounds
    bounds = (minx, miny, maxx, maxy)
    parcel_box = _box(*bounds)
    hit_a = idx_a[idx_a.intersects(parcel_box)]
    hit_b = idx_b[idx_b.intersects(parcel_box)]
    if hit_a.empty or hit_b.empty:
        return np.zeros((0, 0, 3), np.uint8), np.zeros((0, 0, 3), np.uint8)
    try:
        a = _window_read(hit_a.iloc[0]["url"], bounds)
        b = _window_read(hit_b.iloc[0]["url"], bounds)
    except Exception:
        return np.zeros((0, 0, 3), np.uint8), np.zeros((0, 0, 3), np.uint8)
    h = min(a.shape[0], b.shape[0]); w = min(a.shape[1], b.shape[1])
    return a[:h, :w], b[:h, :w]


def _rgb_change_features(a, b):
    if a.size == 0 or b.size == 0 or a.shape != b.shape:
        return {"mean_abs_diff": np.nan, "p95_abs_diff": np.nan,
                "ssim": np.nan, "dark_new_frac": np.nan, "pixel_count": 0}
    af = a.astype(np.float32) / 255.0
    bf = b.astype(np.float32) / 255.0
    abs_diff = np.abs(af - bf).mean(axis=-1)
    try:
        from skimage.metrics import structural_similarity as ssim
        s = float(ssim(af.mean(-1), bf.mean(-1), data_range=1.0))
    except Exception:
        s = float("nan")
    lum_a = af.mean(-1); lum_b = bf.mean(-1)
    return {
        "mean_abs_diff": float(abs_diff.mean()),
        "p95_abs_diff":  float(np.percentile(abs_diff, 95)),
        "ssim":          s,
        "dark_new_frac": float(((lum_a > 0.30) & (lum_b < 0.15)).mean()),
        "pixel_count":   int(af.shape[0] * af.shape[1]),
    }


def per_parcel_rgb_change(parcels, idx_a, idx_b):
    parcels = parcels.to_crs(PHILLY_CRS).reset_index(drop=True)
    feats = []
    for _, row in tqdm(parcels.iterrows(), total=len(parcels), desc="RGB diff"):
        a, b = _chip_pair(row.geometry, idx_a, idx_b)
        feats.append(_rgb_change_features(a, b))
    feats_df = pd.DataFrame(feats)
    return gpd.GeoDataFrame(
        pd.concat([parcels, feats_df], axis=1),
        geometry=parcels.geometry, crs=PHILLY_CRS,
    )


def flag_rgb_collapses(features, min_mean_abs_diff=0.18,
                       min_dark_new_frac=0.05, max_ssim=0.55):
    f = features.copy()
    cond = (
        (f["mean_abs_diff"].fillna(0) >= min_mean_abs_diff)
        & ((f["dark_new_frac"].fillna(0) >= min_dark_new_frac)
           | (f["ssim"].fillna(1.0) <= max_ssim))
    )
    flagged = f.loc[cond].copy()
    flagged["change_score"] = (
        flagged["mean_abs_diff"].fillna(0) * 0.5
        + flagged["dark_new_frac"].fillna(0) * 0.3
        + (1 - flagged["ssim"].fillna(1.0)) * 0.2
    )
    return flagged.sort_values("change_score", ascending=False)


# --- Wipe stale caches that used the broken URL pattern ---------------------
for _p in [
    DRIVE_BASE / "data" / "rgb_change_balanced.parquet",
    DRIVE_BASE / "data" / "rgb_change_natural.parquet",
]:
    if _p.exists():
        print(f"Removing stale cache: {_p}")
        _p.unlink()


# --- Load tile indices ------------------------------------------------------
print("Loading 2024 tile index from source.coop …")
tiles_2024 = load_tile_index(2024)
print(f"  {len(tiles_2024)} tiles")

print("Loading 2025 tile index from source.coop …")
tiles_2025 = load_tile_index(2025)
print(f"  {len(tiles_2025)} tiles")

In [ ]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# HTTP multiplexing + VSI cache (safe to re-run)
os.environ["VSI_CACHE"] = "TRUE"
os.environ["VSI_CACHE_SIZE"] = "200000000"
os.environ["CPL_VSIL_CURL_CACHE_SIZE"] = "200000000"
os.environ["GDAL_HTTP_MULTIPLEX"] = "YES"
os.environ["GDAL_HTTP_VERSION"] = "2"

_NAN_FEAT = {"mean_abs_diff": np.nan, "p95_abs_diff": np.nan,
             "ssim": np.nan, "dark_new_frac": np.nan, "pixel_count": 0}


def _process_tile_pair(url_a, url_b, rows):
    """Open one (url_a, url_b) pair, read every parcel window in `rows`,
    return {pidx: features}. Each thread holds its own rasterio datasets."""
    out = {}
    try:
        src_a = rasterio.open(f"/vsicurl/{url_a}")
        src_b = rasterio.open(f"/vsicurl/{url_b}")
    except Exception:
        return {int(r["pidx"]): _NAN_FEAT for _, r in rows.iterrows()}
    try:
        for _, row in rows.iterrows():
            pidx = int(row["pidx"])
            b_ = row["bounds"]
            try:
                wa = src_a.window(*b_); wb = src_b.window(*b_)
                a = np.transpose(
                    src_a.read([1,2,3], window=wa, boundless=True, fill_value=0),
                    (1,2,0),
                )
                b = np.transpose(
                    src_b.read([1,2,3], window=wb, boundless=True, fill_value=0),
                    (1,2,0),
                )
                h = min(a.shape[0], b.shape[0]); w = min(a.shape[1], b.shape[1])
                out[pidx] = _rgb_change_features(a[:h, :w], b[:h, :w])
            except Exception:
                out[pidx] = _NAN_FEAT
    finally:
        src_a.close(); src_b.close()
    return out


def per_parcel_rgb_change(parcels, idx_a, idx_b, max_workers=12):
    """Threaded version — ~10× faster than the single-threaded tile batching.
    max_workers=12 is a good default; push to 16-24 if Cloudflare tolerates it.
    """
    parcels = parcels.to_crs(PHILLY_CRS).reset_index(drop=True)
    n = len(parcels)
    feats = [None] * n

    parbuf = gpd.GeoDataFrame(
        {"pidx": np.arange(n), "geometry": parcels.geometry.buffer(5.0)},
        crs=PHILLY_CRS,
    )
    sj_a = gpd.sjoin(
        parbuf, idx_a[["url", "geometry"]].rename(columns={"url": "url_a"}),
        how="left", predicate="intersects",
    ).drop_duplicates(subset="pidx", keep="first")[["pidx", "url_a"]]
    sj_b = gpd.sjoin(
        parbuf, idx_b[["url", "geometry"]].rename(columns={"url": "url_b"}),
        how="left", predicate="intersects",
    ).drop_duplicates(subset="pidx", keep="first")[["pidx", "url_b"]]
    plan = sj_a.merge(sj_b, on="pidx", how="outer")
    plan["bounds"] = [parbuf.geometry.iloc[i].bounds for i in plan["pidx"]]

    # Parcels with no covering tile in either year: mark NaN directly
    missing = plan[plan["url_a"].isna() | plan["url_b"].isna()]
    for pidx in missing["pidx"]:
        feats[int(pidx)] = _NAN_FEAT
    plan = plan.dropna(subset=["url_a", "url_b"])

    groups = list(plan.groupby(["url_a", "url_b"]))
    print(f"{len(groups)} tile pairs over {len(plan)} parcels "
          f"(avg {len(plan)/max(len(groups),1):.1f} parcels/pair)")

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [
            ex.submit(_process_tile_pair, url_a, url_b, grp)
            for (url_a, url_b), grp in groups
        ]
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Tile pairs"):
            for pidx, f in fut.result().items():
                feats[pidx] = f

    feats_df = pd.DataFrame(feats)
    return gpd.GeoDataFrame(
        pd.concat([parcels, feats_df], axis=1),
        geometry=parcels.geometry, crs=PHILLY_CRS,
    )

In [ ]:
BAL_CHANGE_PATH = DRIVE_BASE / "data" / "rgb_change_balanced.parquet"

if BAL_CHANGE_PATH.exists():
    bal_change = pd.read_parquet(BAL_CHANGE_PATH)
    print(f"Loaded cached: {BAL_CHANGE_PATH}  ({len(bal_change):,} rows)")
else:
    print("Extracting 2024/2025 change features for balanced_parcels "
          f"({len(balanced_parcels):,} parcels, ~10–20 min) …")
    bal_change_gdf = per_parcel_rgb_change(balanced_parcels, tiles_2024, tiles_2025)
    bal_change = pd.DataFrame(bal_change_gdf.drop(columns="geometry"))
    bal_change.to_parquet(BAL_CHANGE_PATH)
    print(f"Saved: {BAL_CHANGE_PATH}")

CHANGE_COLS = ["mean_abs_diff", "p95_abs_diff", "ssim", "dark_new_frac"]
X_change = bal_change[CHANGE_COLS].to_numpy(dtype=np.float32)
coverage_mask = bal_change["pixel_count"].to_numpy() > 0

print(f"\nCoverage: {coverage_mask.sum():,} / {len(bal_change):,} parcels "
      f"have both-year imagery ({coverage_mask.mean():.1%})")
print(f"NaN rate by feature: "
      f"{dict(bal_change[CHANGE_COLS].isna().mean().round(4))}")

# Impute NaN with column median so downstream classifiers don't crash
for c in CHANGE_COLS:
    X_change[np.isnan(X_change[:, CHANGE_COLS.index(c)]), CHANGE_COLS.index(c)] = (
        np.nanmedian(X_change[:, CHANGE_COLS.index(c)])
    )

In [ ]:
# Drop duplicated column names (keep first occurrence of each)
bal_change = bal_change.loc[:, ~bal_change.columns.duplicated()]
print(f"Columns after dedupe: {list(bal_change.columns)}")

bal_change.to_parquet(BAL_CHANGE_PATH)
print(f"Saved: {BAL_CHANGE_PATH}")

In [ ]:
CHANGE_COLS = ["mean_abs_diff", "p95_abs_diff", "ssim", "dark_new_frac"]

fig, axs = plt.subplots(1, 4, figsize=(18, 3.5))
for ax, col in zip(axs, CHANGE_COLS):
    x0 = bal_change.loc[y_all == 0, col].dropna()
    x1 = bal_change.loc[y_all == 1, col].dropna()
    bins = np.linspace(
        np.nanpercentile(bal_change[col], 1),
        np.nanpercentile(bal_change[col], 99),
        40,
    )
    ax.hist(x0, bins=bins, alpha=0.5, density=True, label="Stable",     color="#4a7cbf")
    ax.hist(x1, bins=bins, alpha=0.5, density=True, label="Distressed", color="#d62728")
    ax.set_title(col); ax.set_yticks([]); ax.legend(fontsize=8)
fig.suptitle("2024 → 2025 change features, split by distress label", y=1.02)
plt.tight_layout(); plt.show()

from sklearn.metrics import roc_auc_score
print("\nSingle-feature AUCs (discrimination power of each change metric):")
for col in CHANGE_COLS:
    v = bal_change[col].fillna(bal_change[col].median()).to_numpy()
    direction = 1.0 if col != "ssim" else -1.0
    auc = roc_auc_score(y_all, direction * v)
    print(f"  {col:>14s}:  AUC = {auc:.3f}")

## 11d. Does adding change features help the Clay classifier?

Three-way comparison on the same train/val split:

1. **Clay only** — 3072-dim multi-token embeddings (baseline, already trained above)
2. **Change only** — 4 change features, LogisticRegression
3. **Combined** — Clay ⊕ scaled change features, LogisticRegression

Metric is average precision (AP) on the balanced-val set AND the natural-prior holdout. If Combined beats Clay-only by a non-trivial margin on the **natural-prior** holdout, the change features are carrying real signal.

In [ ]:
CHANGE_COLS = ["mean_abs_diff", "p95_abs_diff", "ssim", "dark_new_frac"]
X_change = bal_change[CHANGE_COLS].to_numpy(dtype=np.float32)
for j in range(X_change.shape[1]):
    X_change[np.isnan(X_change[:, j]), j] = np.nanmedian(X_change[:, j])
print(f"X_change shape: {X_change.shape}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score

# Reuse the same val indices computed earlier
X_change_train, X_change_val = X_change[idx_all != idx_val_pos[0]], X_change  # placeholder
# Re-derive the split cleanly:
from sklearn.model_selection import train_test_split as _tts
idx_train_pos, idx_val_pos_chk = _tts(
    np.arange(len(balanced_parcels)), test_size=0.2, stratify=y_all, random_state=SEED,
)[:2]
assert np.array_equal(np.sort(idx_val_pos_chk), np.sort(idx_val_pos))

X_change_train = X_change[idx_train_pos]
X_change_val   = X_change[idx_val_pos]

def _fit_eval(Xtr, ytr, Xva, yva, name):
    pipe = Pipeline([
        ("scale", StandardScaler(with_mean=True, with_std=True)),
        ("clf",   LogisticRegression(C=1.0, class_weight="balanced",
                                     max_iter=2000, random_state=SEED)),
    ])
    pipe.fit(Xtr, ytr)
    proba = pipe.predict_proba(Xva)[:, 1]
    ap = average_precision_score(yva, proba)
    print(f"  {name:<20s}  val AP = {ap:.4f}")
    return pipe, proba, ap

print("Validation AP comparison:")
clf_clay,    proba_clay,    ap_clay    = _fit_eval(X_train,                        y_train, X_val,                        y_val, "Clay only (3072-d)")
clf_change,  proba_change,  ap_change  = _fit_eval(X_change_train,                 y_train, X_change_val,                 y_val, "Change only (4-d)")
X_combo_train = np.hstack([X_train, X_change_train])
X_combo_val   = np.hstack([X_val,   X_change_val])
clf_combo,   proba_combo,   ap_combo   = _fit_eval(X_combo_train,                  y_train, X_combo_val,                  y_val, "Combined (3076-d)")

# Precision-recall curves
from sklearn.metrics import precision_recall_curve
fig, ax = plt.subplots(figsize=(7, 5))
for name, proba, color in [
    ("Clay only",    proba_clay,    "#4a7cbf"),
    ("Change only",  proba_change,  "#ff7f0e"),
    ("Combined",     proba_combo,   "#2ca02c"),
]:
    p, r, _ = precision_recall_curve(y_val, proba)
    ax.plot(r, p, label=f"{name} (AP={average_precision_score(y_val, proba):.3f})",
            color=color)
ax.axhline(y_val.mean(), ls="--", color="grey", alpha=0.6,
           label=f"Balanced prior = {y_val.mean():.2f}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Val PR curves: Clay vs Change vs Combined")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
NAT_CHANGE_PATH = DRIVE_BASE / "data" / "rgb_change_natural.parquet"

if NAT_CHANGE_PATH.exists():
    nat_change = pd.read_parquet(NAT_CHANGE_PATH)
else:
    print(f"Extracting change features for natural-dist holdout "
          f"({len(natural_test):,} parcels) …")
    nat_change_gdf = per_parcel_rgb_change(natural_test, tiles_2024, tiles_2025)
    nat_change = pd.DataFrame(nat_change_gdf.drop(columns="geometry"))
    nat_change = nat_change.loc[:, ~nat_change.columns.duplicated()]
    nat_change.to_parquet(NAT_CHANGE_PATH)

X_change_nat = nat_change[CHANGE_COLS].to_numpy(dtype=np.float32)
for j in range(X_change_nat.shape[1]):
    X_change_nat[np.isnan(X_change_nat[:, j]), j] = np.nanmedian(X_change_nat[:, j])

X_combo_nat = np.hstack([X_nat, X_change_nat])

print(f"Natural-prior holdout AP (true prior = {y_nat.mean():.4f}):")
for name, pipe, X in [
    ("Clay only",    clf_clay,   X_nat),
    ("Change only",  clf_change, X_change_nat),
    ("Combined",     clf_combo,  X_combo_nat),
]:
    proba = pipe.predict_proba(X)[:, 1]
    ap    = average_precision_score(y_nat, proba)
    # Lift over the natural prior
    lift = ap / y_nat.mean()
    print(f"  {name:<20s}  AP = {ap:.4f}  (lift vs prior: {lift:.1f}×)")

## 11e. Standalone roof-collapse flag (independent of the distress classifier)

Whether or not change features help the distress classifier, they enable a
separate task the Clay pipeline can't do: detect parcels that *physically
changed* between April 2024 and 2025. These are collapse / demolition /
major-repair candidates regardless of the noisy "distressed" label.

In [ ]:
full_parcels_with_change = pd.concat(
    [balanced_parcels.reset_index(drop=True),
     bal_change[CHANGE_COLS + ["pixel_count"]].reset_index(drop=True)],
    axis=1,
)
full_parcels_with_change = gpd.GeoDataFrame(
    full_parcels_with_change, geometry="geometry", crs=PHILLY_CRS
) if False else full_parcels_with_change  # already has geometry

flagged_rgb = flag_rgb_collapses(
    full_parcels_with_change,
    min_mean_abs_diff=0.18, min_dark_new_frac=0.05, max_ssim=0.55,
)
print(f"RGB-change-flagged parcels (balanced pool): {len(flagged_rgb):,}")
print(f"  of which labeled distressed: "
      f"{(flagged_rgb['label'] > 0).sum()} / {len(flagged_rgb)} "
      f"({(flagged_rgb['label'] > 0).mean():.1%})")

# Show top-12 flagged chips using the same display helpers as section 11b
top_ids  = flagged_rgb.head(12).index.to_numpy()
top_scrs = flagged_rgb.head(12)["change_score"].to_numpy()

fig, axs = plt.subplots(2, 6, figsize=(15, 5.5))
for ax, pid, sc in zip(axs.flatten(), top_ids, top_scrs):
    try:
        ax.imshow(load_display_chip(pid))
    except Exception:
        ax.text(0.5, 0.5, "missing", ha="center", va="center")
    ax.set_title(f"#{pid}  score={sc:.2f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_edgecolor("#d62728"); s.set_linewidth(3)
fig.suptitle("Top RGB-change-flagged parcels (collapse candidates)", y=1.02)
plt.tight_layout(); plt.show()

## 12. Feature importance — which dimensions matter

In [ ]:
importances = rf.feature_importances_
top = np.argsort(importances)[::-1][:20]

print("Top 20 most important Clay embedding dimensions:")
print(f"{'rank':>4}  {'dim':>4}  {'importance':>10}")
for i, d in enumerate(top):
    print(f"{i+1:4d}  {d:4d}  {importances[d]:10.4f}")

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(len(importances)), sorted(importances, reverse=True))
ax.set_xlabel("Rank")
ax.set_ylabel("Importance")
ax.set_title("Clay embedding dimension importances (sorted)")
plt.tight_layout()
plt.show()

## 13. Save the fitted RandomForest

In [ ]:
import joblib

RF_PATH = DRIVE_BASE / "data" / "clay_rf_classifier.joblib"
joblib.dump(rf, RF_PATH)
print(f"Saved RandomForest → {RF_PATH}")
print(f"  size: {RF_PATH.stat().st_size / (1024*1024):.1f} MB")
print(f"  threshold: {THRESHOLD_TUNED:.3f}")

## 14. Full-city inference (OPTIONAL)

Gated behind `RUN_FULL_CITY = False`. Precaches ~547K chips (~30 min),
extracts Clay embeddings for all of them (~30 min on A100), then runs
RandomForest inference (~1 min).

In [ ]:
RUN_FULL_CITY = True

if not RUN_FULL_CITY:
    print("Skipping full-city inference. Set RUN_FULL_CITY=True to run.")
else:
    full_chip_dir = Path("/content/full_chips") if IS_COLAB else PROJECT_ROOT / "data" / "full_chips"
    full_chip_dir.mkdir(parents=True, exist_ok=True)

    full_df = parcels_labeled.copy().reset_index(drop=True)
    full_df['binary_label'] = (full_df['label'] > 0).astype(int)

    print(f"Chipping {len(full_df):,} parcels (~30 min)…")
    precache_chips(full_df, src, full_chip_dir, desc="Full-city chips")

    FULL_EMBED_PATH = DRIVE_BASE / "data" / "clay_embeddings_fullcity.npz"

    if FULL_EMBED_PATH.exists():
        data = np.load(FULL_EMBED_PATH)
        X_full = data['X']
    else:
        full_loader = DataLoader(
            ParcelDataset(full_df, chip_dir=full_chip_dir, label_col='binary_label'),
            batch_size=64, shuffle=False, num_workers=0,
            pin_memory=(DEVICE == 'cuda'),
        )
        print(f"Extracting Clay embeddings for {len(full_df):,} parcels …")
        X_full, _ = extract_embeddings(full_loader, clay_encoder)
        np.savez(FULL_EMBED_PATH, X=X_full)

    distress_proba = rf.predict_proba(X_full)[:, 1]
    full_df["distress_score"]  = distress_proba
    full_df["pred_distressed"] = (distress_proba > THRESHOLD_TUNED).astype(int)

    # Permit filter
    full_df.loc[full_df["label_permit_flagged"], "pred_distressed"] = 0

    print("\nPrediction counts after permit filter:")
    print(full_df["pred_distressed"].value_counts())

    out_path = DRIVE_BASE / "output" / "predictions.geojson"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    cols = ["geometry", "parcel_id", "distress_score", "pred_distressed", "label"]
    full_df[cols].to_file(out_path, driver="GeoJSON")
    print(f"\nSaved: {out_path}")